# 单因素方差分析完整教程：F 检验与事后比较

## 📚 教学目标
1. 理解方差分析（ANOVA）的目的和适用条件
2. 掌握 F 检验的原理（组间变异 vs 组内变异）
3. 手算 F 统量并理解其含义
4. 用 scipy.stats.f_oneway 验证手算结果
5. 理解事后比较（Bonferroni 校正）的方法

**参考书**：《基础统计学(第14版)》（Triola）第 12-1 节
**教学策略**：从药物疗效的简单例子入手，逐步理解 ANOVA 的完整流程

---

## 1. 场景设定：三种药物的疗效比较

### 🎯 问题

研究人员比较三种药物对血压降低的效果。将 30 名患者随机分配到三组：
- 药物 A: 10 名患者
- 药物 B: 10 名患者
- 药物 C: 10 名患者

问题：三种药物的平均血压降低量是否相同？

### 📖 书中的定义

> 方差分析（ANOVA）用于检验三个或更多独立样本的均值是否相等。
> 零假设 H₀: μ₁ = μ₂ = μ₃ = ... = μₖ
> 备择假设 H₁: 至少有一对均值不等

### 📐 F 统计量

$$F = \frac{MS_{between}}{MS_{within}} = \frac{SS_{between} / (k-1)}{SS_{within} / (n-k)}$$

其中：
- $SS_{between}$ = 组间平方和（不同组之间的变异）
- $SS_{within}$ = 组内平方和（组内的随机变异）
- k = 组数
- n = 总样本量

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 生成数据与探索

### 📐 数据生成过程 (DGP)

- 药物 A: μ=10, σ=3 (中等效果)
- 药物 B: μ=15, σ=3 (最佳效果)
- 药物 C: μ=8, σ=3 (最差效果)

In [ ]:
# ========== 步骤 1: 生成数据 ==========

print("=" * 60)
print("📋 三种药物的血压降低量")
print("=" * 60)

# --- 数据生成过程 (DGP) ---
n_per_group = 10
drug_a = np.random.normal(10, 3, n_per_group)
drug_b = np.random.normal(15, 3, n_per_group)
drug_c = np.random.normal(8, 3, n_per_group)

# 合并数据
all_data = np.concatenate([drug_a, drug_b, drug_c])
groups = ['A'] * n_per_group + ['B'] * n_per_group + ['C'] * n_per_group

print(f"\n📊 数据概况:")
print(f"{'药物':<6} {'n':<6} {'均值':<10} {'标准差':<10}")
print("-" * 32)
for name, data in [('A', drug_a), ('B', drug_b), ('C', drug_c)]:
    print(f"  {name:<6} {len(data):<6} {np.mean(data):<10.4f} {np.std(data, ddof=1):<10.4f}")

print(f"\n  总样本量 n = {len(all_data)}")
print(f"  总均值 = {np.mean(all_data):.4f}")

In [ ]:
# ========== 可视化: 箱形图 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形图
ax1 = axes[0]
bp = ax1.boxplot([drug_a, drug_b, drug_c], labels=['Drug A', 'Drug B', 'Drug C'],
                patch_artist=True)
colors = ['#2ecc71', '#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Blood Pressure Reduction', fontsize=12)
ax1.set_title('Drug Efficacy Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 散点图 + 均值
ax2 = axes[1]
for i, (name, data, color) in enumerate(zip(['A', 'B', 'C'], [drug_a, drug_b, drug_c], colors)):
    ax2.scatter([i+1]*len(data), data, alpha=0.5, color=color, s=40)
    ax2.scatter(i+1, np.mean(data), color='black', s=200, marker='D', zorder=5)
ax2.set_xticks([1, 2, 3])
ax2.set_xticklabels(['Drug A', 'Drug B', 'Drug C'])
ax2.set_ylabel('Blood Pressure Reduction', fontsize=12)
ax2.set_title('Individual Data Points with Means', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：箱形图显示各组的中位数、四分位数和异常值")
print("  右图：散点图显示每个数据点，菱形标记为均值")
print("  从图中看，药物 B 的效果似乎最好")

---

## 3. 手算 ANOVA

### 📐 平方和分解

$$SS_{total} = SS_{between} + SS_{within}$$

$$SS_{total} = \sum (x_i - \bar{x})^2$$

$$SS_{between} = \sum n_j (\bar{x}_j - \bar{x})^2$$

$$SS_{within} = \sum \sum (x_{ij} - \bar{x}_j)^2$$

In [ ]:
# ========== 步骤 2: 手算 ANOVA ==========

print("=" * 60)
print("📋 手算单因素方差分析")
print("=" * 60)

k = 3  # 组数
n_total = len(all_data)
grand_mean = np.mean(all_data)

print(f"\n📊 基本统计量:")
print(f"  组数 k = {k}")
print(f"  总样本量 n = {n_total}")
print(f"  总均值 x̄ = {grand_mean:.4f}")

# 组均值
group_means = [np.mean(drug_a), np.mean(drug_b), np.mean(drug_c)]
group_names = ['A', 'B', 'C']

print(f"\n📊 组均值:")
for name, mean in zip(group_names, group_means):
    print(f"  药物 {name}: x̄ = {mean:.4f}")

# SS_between
ss_between = sum(n_per_group * (m - grand_mean)**2 for m in group_means)

# SS_within
ss_within = 0
for data in [drug_a, drug_b, drug_c]:
    group_mean = np.mean(data)
    ss_within += np.sum((data - group_mean)**2)

# SS_total
ss_total = np.sum((all_data - grand_mean)**2)

print(f"\n📊 平方和:")
print(f"  SS_between (组间) = {ss_between:.4f}")
print(f"  SS_within (组内) = {ss_within:.4f}")
print(f"  SS_total (总) = {ss_total:.4f}")
print(f"  验证: SS_between + SS_within = {ss_between + ss_within:.4f}")

# 自由度
df_between = k - 1
df_within = n_total - k
df_total = n_total - 1

print(f"\n📊 自由度:")
print(f"  df_between = k - 1 = {df_between}")
print(f"  df_within = n - k = {df_within}")
print(f"  df_total = n - 1 = {df_total}")

# 均方
ms_between = ss_between / df_between
ms_within = ss_within / df_within

print(f"\n📊 均方:")
print(f"  MS_between = SS_between / df_between = {ms_between:.4f}")
print(f"  MS_within = SS_within / df_within = {ms_within:.4f}")

# F 统计量
f_stat_manual = ms_between / ms_within

print(f"\n📊 F 统计量:")
print(f"  F = MS_between / MS_within = {f_stat_manual:.4f}")

# p 值
p_value = 1 - stats.f.cdf(f_stat_manual, df_between, df_within)

print(f"\n📊 p 值:")
print(f"  p = {p_value:.6f}")

# 判断
alpha = 0.05
if p_value < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：至少有一组均值与其他组不同")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀：没有足够证据说明均值不同")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 3: ANOVA 表 ==========

print("=" * 60)
print("📋 方差分析表 (ANOVA Table)")
print("=" * 60)

print(f"\n{'变异来源':<12} {'SS':<12} {'df':<8} {'MS':<12} {'F':<10} {'p 值':<12}")
print("-" * 66)
print(f"  {'组间':<10} {ss_between:<12.4f} {df_between:<8} {ms_between:<12.4f} {f_stat_manual:<10.4f} {p_value:<12.6f}")
print(f"  {'组内':<10} {ss_within:<12.4f} {df_within:<8} {ms_within:<12.4f}")
print(f"  {'合计':<10} {ss_total:<12.4f} {df_total:<8}")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 4: scipy 验证 ==========

print("=" * 60)
print("🔬 scipy.stats.f_oneway 验证")
print("=" * 60)

f_scipy, p_scipy = stats.f_oneway(drug_a, drug_b, drug_c)

print(f"\n📊 手算 vs scipy 对比:")
print(f"  手算 F = {f_stat_manual:.6f}")
print(f"  scipy F = {f_scipy:.6f}")
print(f"  手算 p = {p_value:.6f}")
print(f"  scipy p = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

print("\n" + "=" * 60)

---

## 4. 事后比较：Tukey HSD

### 📖 书中的要点

> 当 ANOVA 拒绝 H₀ 时，我们需要知道具体哪些组之间存在差异。
> 事后比较（post hoc comparison）用于检验所有成对均值的差异。

### 📐 Tukey HSD 检验

$$q = \frac{\bar{x}_i - \bar{x}_j}{\sqrt{MS_{within} / n}}$$

### 📐 Bonferroni 校正

当进行多次比较时，需要调整显著性水平以控制总体错误率：
$$\alpha_{adjusted} = \frac{\alpha}{m}$$

其中 m 是比较次数。

In [ ]:
# ========== 步骤 5: 事后比较 ==========

print("=" * 60)
print("📋 事后比较: Tukey HSD 检验")
print("=" * 60)

# Tukey HSD
tukey = pairwise_tukeyhsd(all_data, groups, alpha=0.05)
print(f"\n📊 Tukey HSD 检验结果:")
print(tukey)

# 手算 Bonferroni 校正
m = 3  # 比较次数 (C(3,2) = 3)
alpha_bonferroni = 0.05 / m

print(f"\n📊 Bonferroni 校正:")
print(f"  原始 α = 0.05")
print(f"  比较次数 m = {m}")
print(f"  校正后 α = {alpha_bonferroni:.4f}")

# 手算两两比较
print(f"\n📊 两两 t 检验 (Bonferroni 校正):")
pairs = [('A', 'B', drug_a, drug_b), ('A', 'C', drug_a, drug_c), ('B', 'C', drug_b, drug_c)]
for name1, name2, data1, data2 in pairs:
    t_stat, p_val = stats.ttest_ind(data1, data2)
    sig = '显著' if p_val < alpha_bonferroni else '不显著'
    print(f"  药物 {name1} vs {name2}: t = {t_stat:.4f}, p = {p_val:.4f} ({sig})")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 事后比较 ==========

fig, ax = plt.subplots(figsize=(8, 6))

# 均值和置信区间
means = [np.mean(d) for d in [drug_a, drug_b, drug_c]]
sems = [stats.sem(d) for d in [drug_a, drug_b, drug_c]]
ci95 = [s * stats.t.ppf(0.975, len(d)-1) for s, d in zip(sems, [drug_a, drug_b, drug_c])]

x_pos = np.arange(3)
ax.errorbar(x_pos, means, yerr=ci95, fmt='o', color='steelblue', 
            markersize=10, capsize=8, linewidth=2, capthick=2)

ax.set_xticks(x_pos)
ax.set_xticklabels(['Drug A', 'Drug B', 'Drug C'])
ax.set_ylabel('Blood Pressure Reduction (Mean ± 95% CI)', fontsize=12)
ax.set_title('Group Means with 95% Confidence Intervals', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 标注显著性
y_max = max(m + c for m, c in zip(means, ci95))
ax.plot([0, 1], [y_max + 1, y_max + 1], 'k-', linewidth=1.5)
ax.text(0.5, y_max + 1.5, '***', ha='center', fontsize=14)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  误差线表示 95% 置信区间")
print("  如果两个组的置信区间不重叠，通常表示差异显著")
print("  *** 表示 p < 0.001 (高度显著)")

---

## 5. ANOVA 条件检查

### 📐 三个必要条件

| 条件 | 检验方法 |
|------|----------|
| 独立性 | 实验设计保证 |
| 正态性 | Shapiro-Wilk 检验 |
| 方差齐性 | Levene 检验 |

In [ ]:
# ========== 步骤 6: 条件检查 ==========

print("=" * 60)
print("📋 ANOVA 条件检查")
print("=" * 60)

# 正态性检验
print(f"\n📊 正态性检验 (Shapiro-Wilk):")
for name, data in [('A', drug_a), ('B', drug_b), ('C', drug_c)]:
    w_stat, p_val = stats.shapiro(data)
    result = '正态' if p_val > 0.05 else '非正态'
    print(f"  药物 {name}: W = {w_stat:.4f}, p = {p_val:.4f} ({result})")

# 方差齐性检验
print(f"\n📊 方差齐性检验 (Levene):")
levene_stat, levene_p = stats.levene(drug_a, drug_b, drug_c)
result = '方差齐性' if levene_p > 0.05 else '方差不齐'
print(f"  Levene 统计量 = {levene_stat:.4f}")
print(f"  p = {levene_p:.4f} ({result})")

print(f"\n💡 解读:")
print(f"  如果正态性检验不通过，应使用非参数方法 (Kruskal-Wallis)")
print(f"  如果方差不齐，应使用 Welch's ANOVA")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 方差分析 (ANOVA)
- **定义**: 检验三个或更多独立样本的均值是否相等
- **公式**: $F = \frac{MS_{between}}{MS_{within}}$
- **含义**: F 值越大，组间差异相对于组内差异越大
- **判断标准**: p < α 时拒绝 H₀

### 📌 平方和分解
- **SS_total**: 总变异
- **SS_between**: 组间变异（可解释的变异）
- **SS_within**: 组内变异（随机误差）
- **关系**: SS_total = SS_between + SS_within

### 📌 F 分布
- **定义**: 两个卡方变量之比的分布
- **自由度**: df₁ = k-1, df₂ = n-k
- **性质**: F 值总是 ≥ 0

### 📌 事后比较
- **目的**: ANOVA 拒绝 H₀ 后，确定具体哪些组不同
- **方法**: Tukey HSD, Bonferroni 校正
- **注意**: 多重比较需要校正以控制错误率

### 🔗 完整流程
```
设定假设 → 检查条件 → 计算 F → 求 p 值 → 判断 → 事后比较
    ↓          ↓         ↓        ↓       ↓       ↓
  H₀/H₁   正态/齐性   公式    查表/软件  p<α?   Tukey HSD
```

---

## ❌ 常见误区

### ❌ 误区 1: 用多个 t 检验代替 ANOVA
**✓ 正确做法**: 当比较三个或更多组时，应使用 ANOVA 而不是多个 t 检验。多个 t 检验会增加第一类错误（假阳性）的概率。ANOVA 通过一次检验控制总体错误率。

### ❌ 误区 2: ANOVA 拒绝 H₀ 后不知道哪些组不同
**✓ 正确做法**: ANOVA 只能告诉你"至少有一组不同"，不能告诉你具体哪些组不同。需要进行事后比较（如 Tukey HSD）来确定具体的差异。

### ❌ 误区 3: 忽略 ANOVA 的前提条件
**✓ 正确做法**: ANOVA 要求数据满足独立性、正态性和方差齐性。如果正态性不满足，应使用 Kruskal-Wallis 检验；如果方差不齐，应使用 Welch's ANOVA。

### ❌ 误区 4: 在不满足参数检验条件时误用参数方法
**✓ 正确做法**: 当数据严重偏离正态分布或方差严重不齐时，应使用非参数方法（如 Kruskal-Wallis 检验）。强行使用 ANOVA 可能导致错误的结论。

### ❌ 误区 5: F 值大就代表效果强
**✓ 正确理解**: F 值的大小受样本量影响。大样本可以使微小差异变得"统计显著"。应结合效应量（如 η²）和实际意义来解释结果。η² = SS_between / SS_total。